# FleetVision Phase 04.5K — Validation Baseline Error Analysis

**核准方案：B — 自動量化分析＋人工複核工作清單**

- Model：YOLOv8s Detect，single class `damage`
- Source model：Phase 04.5J `best.pt`
- Analysis split：**validation only**
- Matching：IoU ≥ 0.50，one-to-one deterministic matching
- Threshold sweep：0.05～0.80
- Test set：已正式評估一次；本 Notebook 不讀取、不解壓、不用於 tuning
- Training：本 Notebook 不啟動任何訓練或 fine-tuning
- Deployment acceptance：`NOT_YET_APPROVED`

**執行方式**

1. 依序執行 Cell 1～Cell 8。
2. 每格必須顯示該格 `PASS` 才能繼續。
3. Cell 8 會產生 `04_5K_*_ZIP_LOG.zip`，回傳該 ZIP 進行外層 Gate 審核。
4. 任何 Gate 驗證失敗都應立即停止，不得自行降低限制。


In [ ]:
# Cell 1
# 操作類型：新增並執行
# 插入或覆蓋位置：Notebook 第一個程式碼儲存格
# 主要目的：掛載 Google Drive，鎖定唯一 04.5J PASS 執行結果，建立 04.5K 唯一輸出目錄。
# 執行前提：04.5J 已完成，best.pt 與 04_5J_gate_result.json 保留在 Google Drive。
# 重點：僅定位 04.5J 證據；不讀取 test metrics，不開始推論或訓練。
# 驗收標準：顯示 CELL 1 PASS、04.5J 根目錄、gate result、best.pt 與本次 SESSION_ROOT。

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path, PurePosixPath
import base64
import csv
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import secrets
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from datetime import datetime, timezone

import pandas as pd

EXPECTED_BEST_PT_SHA256 = '90A880513A42EF2DB1373902D98FF09D1756AB7A8A4EEA6A7AA231D4020B77BF'
EXPECTED_J_CLASSIFICATION = 'CONTROLLED_COLAB_BASELINE_TRAINING_COMPLETED'
EXPECTED_VALID_IMAGES = 168
EXPECTED_VALID_ANNOTATIONS = 325
BASE_DIR_OVERRIDE = ''  # 只有自動定位失敗時，才填入 04_5J 絕對路徑。
RUN_RESULT_OVERRIDE = ''  # 只有存在多個 gate result 時，才填入 execution_id 或完整檔名片段。

base_candidates = [
    Path('/content/drive/MyDrive/AI_Class/00.Project/FleetVision/04_5J'),
    Path('/content/drive/MyDrive/FleetVision/04_5J'),
]
if BASE_DIR_OVERRIDE:
    base_candidates = [Path(BASE_DIR_OVERRIDE)]
base_candidates = [p for p in base_candidates if (p / 'input').is_dir() and (p / 'runs').is_dir()]
if len(base_candidates) != 1:
    raise RuntimeError(f'必須精確定位 1 個 04.5J 根目錄，目前找到：{base_candidates}')
J_BASE = base_candidates[0]

result_candidates = sorted((J_BASE / 'runs').rglob('04_5J_gate_result.json'))
if RUN_RESULT_OVERRIDE:
    result_candidates = [p for p in result_candidates if RUN_RESULT_OVERRIDE in str(p)]
if len(result_candidates) != 1:
    raise RuntimeError(
        '必須精確找到 1 份 04_5J_gate_result.json；目前找到 '
        f'{len(result_candidates)} 份：{[str(p) for p in result_candidates]}'
    )
J_GATE_RESULT_PATH = result_candidates[0]
j_gate = json.loads(J_GATE_RESULT_PATH.read_text(encoding='utf-8'))
required_j = {
    'gate_id': '04.5J',
    'outcome': 'PASS',
    'classification': EXPECTED_J_CLASSIFICATION,
    'best_pt_sha256': EXPECTED_BEST_PT_SHA256,
    'model_training_completed': True,
    'test_evaluation_completed': True,
    'weights_in_zip': False,
}
for key, expected in required_j.items():
    actual = j_gate.get(key)
    if actual != expected:
        raise RuntimeError(f'04.5J gate 欄位不符：{key}={actual!r}，預期 {expected!r}')

BEST_PT = Path(j_gate['best_pt_path'])
if not BEST_PT.is_file():
    raise FileNotFoundError(f'找不到 best.pt：{BEST_PT}')
BUNDLE_ID = j_gate['bundle_id']
EXECUTION_STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
EXECUTION_ID = f'{EXECUTION_STAMP}_{secrets.token_hex(4)}'
K_BASE = J_BASE.parent / '04_5K'
RUNS_DIR = K_BASE / 'runs'
WORK_ROOT = Path('/content/FleetVision_04_5K') / BUNDLE_ID / EXECUTION_ID
SESSION_ROOT = RUNS_DIR / BUNDLE_ID / EXECUTION_ID
if WORK_ROOT.exists() or SESSION_ROOT.exists():
    raise FileExistsError('04.5K 唯一執行路徑已存在，拒絕覆寫')
WORK_ROOT.mkdir(parents=True, exist_ok=False)
SESSION_ROOT.mkdir(parents=True, exist_ok=False)

print('04.5J root：', J_BASE)
print('04.5J gate result：', J_GATE_RESULT_PATH)
print('best.pt：', BEST_PT)
print('04.5K session：', SESSION_ROOT)
print('CELL 1 PASS')


In [ ]:
# Cell 2
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 1 後新增
# 主要目的：驗證 04.5J 權重與 bundle，僅安全解壓 validation 圖片與標註。
# 執行前提：Cell 1 PASS。
# 重點：只允許 dataset/05_yolo/images/valid/ 與 dataset/05_yolo/labels/valid/；train/test 內容不解壓。
# 驗收標準：best.pt SHA256 正確；validation images=168、labels=168、annotations=325；顯示 CELL 2 PASS。

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest().upper()

best_hash = sha256_file(BEST_PT)
if best_hash != EXPECTED_BEST_PT_SHA256:
    raise RuntimeError(f'best.pt SHA256 不符：{best_hash}')

descriptor_candidates = sorted((J_BASE / 'input').glob('04_5J_bundle_descriptor_*.json'))
descriptor_candidates = [p for p in descriptor_candidates if BUNDLE_ID in p.name]
if len(descriptor_candidates) != 1:
    raise RuntimeError(f'找不到唯一 04.5J descriptor：{descriptor_candidates}')
DESCRIPTOR_PATH = descriptor_candidates[0]
descriptor = json.loads(DESCRIPTOR_PATH.read_text(encoding='utf-8'))
for key, expected in {
    'gate_id': '04.5J-PREP',
    'classification': 'CONTROLLED_COLAB_BASELINE_BUNDLE_PREPARED',
    'bundle_id': BUNDLE_ID,
}.items():
    if descriptor.get(key) != expected:
        raise RuntimeError(f'descriptor 欄位不符：{key}')
TAR_PATH = J_BASE / 'input' / descriptor['tar']['filename']
DATASET_MANIFEST_PATH = J_BASE / 'input' / descriptor['dataset_manifest']['filename']
if sha256_file(TAR_PATH) != str(descriptor['tar']['sha256']).upper():
    raise RuntimeError('04.5J dataset TAR SHA256 不符')
if sha256_file(DATASET_MANIFEST_PATH) != str(descriptor['dataset_manifest']['sha256']).upper():
    raise RuntimeError('04.5J dataset manifest SHA256 不符')

EXTRACT_ROOT = WORK_ROOT / 'validation_extract'
VALID_IMAGES_DIR = EXTRACT_ROOT / 'dataset/05_yolo/images/valid'
VALID_LABELS_DIR = EXTRACT_ROOT / 'dataset/05_yolo/labels/valid'
allowed_prefixes = (
    'dataset/05_yolo/images/valid/',
    'dataset/05_yolo/labels/valid/',
)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=False)
extracted_relpaths = []
with tarfile.open(TAR_PATH, mode='r:*') as archive:
    for member in archive.getmembers():
        raw_name = member.name.replace('\\', '/')
        pure = PurePosixPath(raw_name)
        if pure.is_absolute() or '..' in pure.parts:
            raise RuntimeError(f'TAR unsafe path：{raw_name}')
        if member.issym() or member.islnk() or member.isdev():
            raise RuntimeError(f'TAR 禁止連結或特殊檔案：{raw_name}')
        if not member.isfile() or not raw_name.startswith(allowed_prefixes):
            continue
        source = archive.extractfile(member)
        if source is None:
            raise RuntimeError(f'TAR member 無法讀取：{raw_name}')
        destination = EXTRACT_ROOT / Path(*pure.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open('wb') as target:
            shutil.copyfileobj(source, target)
        extracted_relpaths.append(raw_name)

manifest_rows = list(csv.DictReader(DATASET_MANIFEST_PATH.open('r', encoding='utf-8-sig', newline='')))

def _normalize_manifest_to_tar_path(raw_value: str) -> str:
    relative = str(raw_value).replace('\\', '/')
    while relative.startswith('./'):
        relative = relative[2:]

    pure = PurePosixPath(relative)
    if not relative or pure.is_absolute() or '..' in pure.parts:
        raise RuntimeError(f'manifest 含不安全或空白路徑：{raw_value!r}')

    relative = pure.as_posix()
    prefix = "dataset/05_yolo/"
    return relative if relative.startswith(prefix) else prefix + relative

manifest_valid = {}
for row in manifest_rows:
    full_path = _normalize_manifest_to_tar_path(row['relative_path'])
    if not full_path.startswith(allowed_prefixes):
        continue
    if full_path in manifest_valid:
        raise RuntimeError(f'dataset manifest 含重複 validation 路徑：{full_path}')
    manifest_valid[full_path] = row

extracted_set = set(extracted_relpaths)
manifest_set = set(manifest_valid)

if len(extracted_relpaths) != len(extracted_set):
    raise RuntimeError('TAR validation 範圍含重複 member path')

if extracted_set != manifest_set:
    missing = sorted(manifest_set - extracted_set)[:10]
    extra = sorted(extracted_set - manifest_set)[:10]
    raise RuntimeError(
        'validation manifest/extract 不一致，'
        f'missing={missing}, extra={extra}, '
        f'manifest_count={len(manifest_set)}, extracted_count={len(extracted_set)}'
    )
for relative, row in manifest_valid.items():
    path = EXTRACT_ROOT / relative
    if path.stat().st_size != int(row['size_bytes']):
        raise RuntimeError(f'validation file size 不符：{relative}')
    if sha256_file(path) != str(row['sha256']).upper():
        raise RuntimeError(f'validation file SHA256 不符：{relative}')

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
valid_images = sorted(p for p in VALID_IMAGES_DIR.iterdir() if p.suffix.lower() in image_extensions)
valid_labels = sorted(VALID_LABELS_DIR.glob('*.txt'))
annotation_count = 0
for label_path in valid_labels:
    annotation_count += sum(1 for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip())
if len(valid_images) != EXPECTED_VALID_IMAGES:
    raise RuntimeError(f'validation image count 不符：{len(valid_images)}')
if len(valid_labels) != EXPECTED_VALID_IMAGES:
    raise RuntimeError(f'validation label count 不符：{len(valid_labels)}')
if annotation_count != EXPECTED_VALID_ANNOTATIONS:
    raise RuntimeError(f'validation annotation count 不符：{annotation_count}')

print('best.pt SHA256：', best_hash)
print('Validation images / labels / annotations：', len(valid_images), len(valid_labels), annotation_count)
print('CELL 2 PASS')


In [ ]:
# Cell 3
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 2 後新增
# 主要目的：鎖定 Ultralytics 版本，載入經本機測試的 04.5K matching 模組與固定 config。
# 執行前提：Cell 2 PASS。
# 重點：沒有 CUDA 直接停止；只建立分析環境，不呼叫任何訓練 API。
# 驗收標準：Ultralytics=8.4.93、CUDA 可用、config validation PASS、顯示 CELL 3 PASS。

EXPECTED_ULTRALYTICS = '8.4.93'
try:
    current_ultralytics = importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    current_ultralytics = ''
if current_ultralytics != EXPECTED_ULTRALYTICS:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'ultralytics=={EXPECTED_ULTRALYTICS}'],
        check=True,
    )

import torch
if not torch.cuda.is_available():
    raise RuntimeError('04.5K 受控推論要求 CUDA GPU；禁止 CPU fallback')

MODULE_B64 = '...'
CONFIG_B64 = 'Z2F0ZV9pZDogMDQuNUsKCmlucHV0OgogIGFsbG93ZWRfc3BsaXQ6IHZhbGlkCiAgZXhwZWN0ZWRfaW1hZ2VzOiAxNjgKICBleHBlY3RlZF9ncm91bmRfdHJ1dGhfaW5zdGFuY2VzOiAzMjUKICBiZXN0X3B0X3NoYTI1NjogOTBBODgwNTEzQTQyRUYyREIxMzczOTAyRDk4RkYwOUQxNzU2QUI3QThBNEVFQTZBN0FBMjMxRDQwMjBCNzdCRgoKaW5mZXJlbmNlOgogIHVsdHJhbHl0aWNzX3ZlcnNpb246IDguNC45MwogIGNvbmZpZGVuY2VfZmxvb3I6IDAuMDAxCiAgbm1zX2lvdTogMC43MAogIGltZ3N6OiA2NDAKICBtYXhfZGV0OiAzMDAKCm1hdGNoaW5nOgogIGlvdV90aHJlc2hvbGQ6IDAuNTAKICBsb2NhbGl6YXRpb25faW91X21pbjogMC4xMAoKdGhyZXNob2xkczoKICAtIDAuMDUKICAtIDAuMTAKICAtIDAuMTUKICAtIDAuMjAKICAtIDAuMjUKICAtIDAuMzAKICAtIDAuMzUKICAtIDAuNDAKICAtIDAuNDUKICAtIDAuNTAKICAtIDAuNTUKICAtIDAuNjAKICAtIDAuNjUKICAtIDAuNzAKICAtIDAuNzUKICAtIDAuODAKCm9wZXJhdGluZ19wb2ludHM6CiAgaGlnaF9yZWNhbGxfZnJhY3Rpb25fb2ZfbWF4OiAwLjk1CiAgaGlnaF9wcmVjaXNpb25fZnJhY3Rpb25fb2ZfbWF4OiAwLjk1CgpiYm94X3NpemU6CiAgc21hbGxfbWF4X2FyZWFfcmF0aW86IDAuMDEKICBtZWRpdW1fbWF4X2FyZWFfcmF0aW86IDAuMDUKCnJldmlldzoKICBtYXhfb3ZlcmxheV9pbWFnZXM6IDYwCgpzYWZldHk6CiAgZm9yYmlkX3Rlc3RfdHVuaW5nOiB0cnVlCiAgZm9yYmlkX3RyYWluaW5nOiB0cnVlCiAgZGVwbG95bWVudF9hY2NlcHRhbmNlOiBOT1RfWUVUX0FQUFJPVkVECg=='
MODULE_PATH = WORK_ROOT / 'baseline_error_analysis.py'
CONFIG_PATH = WORK_ROOT / 'baseline_error_analysis_config.yaml'
MODULE_PATH.write_bytes(base64.b64decode(MODULE_B64.encode('ascii')))
CONFIG_PATH.write_bytes(base64.b64decode(CONFIG_B64.encode('ascii')))

spec = importlib.util.spec_from_file_location('fleetvision_04_5k_core', MODULE_PATH)
if spec is None or spec.loader is None:
    raise RuntimeError('無法載入 04.5K core module')
core = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = core
spec.loader.exec_module(core)

import yaml
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
core.validate_analysis_config(config)
if core.box_iou_xyxy((0, 0, 10, 10), (0, 0, 10, 10)) != 1.0:
    raise RuntimeError('04.5K core self-test failed')

METRICS_DIR = SESSION_ROOT / 'metrics'
RECORDS_DIR = SESSION_ROOT / 'records'
REPORTS_DIR = SESSION_ROOT / 'reports'
REVIEW_DIR = SESSION_ROOT / 'review'
OVERLAYS_DIR = REVIEW_DIR / 'overlays'
MANIFEST_DIR = SESSION_ROOT / 'manifest'
for directory in [METRICS_DIR, RECORDS_DIR, REPORTS_DIR, REVIEW_DIR, OVERLAYS_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=False)

print('Ultralytics：', importlib.metadata.version('ultralytics'))
print('CUDA device：', torch.cuda.get_device_name(0))
print('Core module SHA256：', sha256_file(MODULE_PATH))
print('Config SHA256：', sha256_file(CONFIG_PATH))
print('CELL 3 PASS')


In [ ]:
# Cell 4
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 3 後新增
# 主要目的：使用 best.pt 對 168 張 validation 圖片執行一次低 confidence floor 推論。
# 執行前提：Cell 3 PASS。
# 重點：固定 conf=0.001、NMS IoU=0.70；後續 threshold sweep 全部離線完成。
# 驗收標準：處理 168 張 validation 圖片，輸出 validation_predictions.csv，顯示 CELL 4 PASS。

from ultralytics import YOLO

model = YOLO(str(BEST_PT))
image_paths = sorted(p for p in VALID_IMAGES_DIR.iterdir() if p.suffix.lower() in image_extensions)
prediction_rows = []
image_shapes = {}
processed_image_ids = []
results = model.predict(
    source=[str(path) for path in image_paths],
    conf=float(config['inference']['confidence_floor']),
    iou=float(config['inference']['nms_iou']),
    imgsz=int(config['inference']['imgsz']),
    max_det=int(config['inference']['max_det']),
    device=0,
    save=False,
    stream=True,
    verbose=False,
)
for result_index, result in enumerate(results):
    if result_index >= len(image_paths):
        raise RuntimeError(
            f'Ultralytics 回傳超過輸入圖片數量：index={result_index}, inputs={len(image_paths)}'
        )

    source_path = image_paths[result_index]
    image_id = source_path.name
    reported_image_id = Path(result.path).name
    expected_synthetic_id = f'image{result_index}.jpg'

    if reported_image_id not in {image_id, expected_synthetic_id}:
        raise RuntimeError(
            'Ultralytics result.path 不符合可驗證映射條件：'
            f'index={result_index}, reported={reported_image_id}, '
            f'original={image_id}, expected_synthetic={expected_synthetic_id}'
        )

    processed_image_ids.append(image_id)
    height, width = (int(result.orig_shape[0]), int(result.orig_shape[1]))
    image_shapes[image_id] = (height, width)
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        continue
    xyxy = boxes.xyxy.detach().cpu().numpy()
    confidences = boxes.conf.detach().cpu().numpy()
    classes = boxes.cls.detach().cpu().numpy().astype(int)
    for index, (coords, confidence, class_id) in enumerate(zip(xyxy, confidences, classes), start=1):
        x1, y1, x2, y2 = (float(value) for value in coords)
        prediction_rows.append({
            'split': 'valid',
            'image_id': image_id,
            'prediction_id': f'{Path(image_id).stem}:pred:{index:04d}',
            'class_id': int(class_id),
            'confidence': float(confidence),
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'bbox_width': x2 - x1,
            'bbox_height': y2 - y1,
            'bbox_area': (x2 - x1) * (y2 - y1),
            'bbox_area_ratio': ((x2 - x1) * (y2 - y1)) / float(width * height),
            'image_width': width,
            'image_height': height,
            'model_sha256': EXPECTED_BEST_PT_SHA256,
        })
if len(processed_image_ids) != len(image_paths):
    raise RuntimeError(
        '推論結果數量未完整覆蓋 validation 圖片：'
        f'processed={len(processed_image_ids)}, expected={len(image_paths)}'
    )

if processed_image_ids != [path.name for path in image_paths]:
    raise RuntimeError('推論結果順序未完整對齊 validation 輸入清單')
PREDICTION_COLUMNS = [
    'split', 'image_id', 'prediction_id', 'class_id', 'confidence',
    'x1', 'y1', 'x2', 'y2', 'bbox_width', 'bbox_height', 'bbox_area',
    'bbox_area_ratio', 'image_width', 'image_height', 'model_sha256',
]
predictions_df = pd.DataFrame(prediction_rows, columns=PREDICTION_COLUMNS)
predictions_df = predictions_df.sort_values(
    ['image_id', 'confidence', 'prediction_id'], ascending=[True, False, True], kind='mergesort'
).reset_index(drop=True)
PREDICTIONS_PATH = RECORDS_DIR / 'validation_predictions.csv'
predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding='utf-8-sig')

print('Validation images processed：', len(processed_image_ids))
print('Raw predictions at confidence floor：', len(predictions_df))
print('Predictions CSV：', PREDICTIONS_PATH)
print('CELL 4 PASS')


In [ ]:
# Cell 5
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 4 後新增
# 主要目的：解析 validation YOLO labels，建立可追蹤 ground-truth CSV。
# 執行前提：Cell 4 PASS。
# 重點：只接受 class 0 damage；驗證正規化座標與 325 個 GT instances。
# 驗收標準：輸出 validation_ground_truth.csv，GT=325，顯示 CELL 5 PASS。

image_by_stem = {path.stem: path for path in image_paths}
ground_truth_rows = []
for label_path in sorted(VALID_LABELS_DIR.glob('*.txt')):
    image_path = image_by_stem.get(label_path.stem)
    if image_path is None:
        raise RuntimeError(f'label 找不到對應 validation image：{label_path.name}')
    image_id = image_path.name
    height, width = image_shapes[image_id]
    lines_in_label = [line.strip() for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    for index, line in enumerate(lines_in_label, start=1):
        parts = line.split()
        if len(parts) != 5:
            raise RuntimeError(f'YOLO label 欄位數錯誤：{label_path.name}:{index}')
        class_id = int(parts[0])
        cx, cy, bw, bh = (float(value) for value in parts[1:])
        if class_id != 0:
            raise RuntimeError(f'非 damage class：{label_path.name}:{index} class={class_id}')
        if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0 and 0.0 < bw <= 1.0 and 0.0 < bh <= 1.0):
            raise RuntimeError(f'YOLO normalized bbox 不合法：{label_path.name}:{index}')
        x1 = (cx - bw / 2.0) * width
        y1 = (cy - bh / 2.0) * height
        x2 = (cx + bw / 2.0) * width
        y2 = (cy + bh / 2.0) * height
        ground_truth_rows.append({
            'split': 'valid',
            'image_id': image_id,
            'gt_id': f'{label_path.stem}:gt:{index:04d}',
            'class_id': class_id,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'bbox_width': x2 - x1,
            'bbox_height': y2 - y1,
            'bbox_area': (x2 - x1) * (y2 - y1),
            'bbox_area_ratio': bw * bh,
            'image_width': width,
            'image_height': height,
        })
GROUND_TRUTH_COLUMNS = [
    'split', 'image_id', 'gt_id', 'class_id', 'x1', 'y1', 'x2', 'y2',
    'bbox_width', 'bbox_height', 'bbox_area', 'bbox_area_ratio',
    'image_width', 'image_height',
]
ground_truth_df = pd.DataFrame(ground_truth_rows, columns=GROUND_TRUTH_COLUMNS)
ground_truth_df = ground_truth_df.sort_values(['image_id', 'gt_id'], kind='mergesort').reset_index(drop=True)
if len(ground_truth_df) != EXPECTED_VALID_ANNOTATIONS:
    raise RuntimeError(f'validation GT count 不符：{len(ground_truth_df)}')
if ground_truth_df['image_id'].nunique() != EXPECTED_VALID_IMAGES:
    raise RuntimeError('validation GT image coverage 不符')
GROUND_TRUTH_PATH = RECORDS_DIR / 'validation_ground_truth.csv'
ground_truth_df.to_csv(GROUND_TRUTH_PATH, index=False, encoding='utf-8-sig')

print('Validation ground-truth instances：', len(ground_truth_df))
print('Ground-truth CSV：', GROUND_TRUTH_PATH)
print('CELL 5 PASS')


In [ ]:
# Cell 6
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 5 後新增
# 主要目的：離線執行 validation threshold sweep，選出三個候選 operating points 並建立逐物件錯誤紀錄。
# 執行前提：Cell 5 PASS。
# 重點：threshold candidates 不是 deployment threshold；matching IoU 固定 0.50。
# 驗收標準：輸出 sweep、operating points、matches、error cases 與圖表，顯示 CELL 6 PASS。

sweep_df = core.run_threshold_sweep(
    predictions_df,
    ground_truth_df,
    thresholds=config['thresholds'],
    iou_threshold=float(config['matching']['iou_threshold']),
    localization_iou_min=float(config['matching']['localization_iou_min']),
)
operating_points_df = core.select_operating_points(
    sweep_df,
    high_recall_fraction_of_max=float(config['operating_points']['high_recall_fraction_of_max']),
    high_precision_fraction_of_max=float(config['operating_points']['high_precision_fraction_of_max']),
)
balanced_threshold = float(
    operating_points_df.loc[operating_points_df['operating_point'] == 'balanced', 'threshold'].iloc[0]
)
detailed = core.evaluate_threshold(
    predictions_df,
    ground_truth_df,
    threshold=balanced_threshold,
    iou_threshold=float(config['matching']['iou_threshold']),
    localization_iou_min=float(config['matching']['localization_iou_min']),
)

small_max = float(config['bbox_size']['small_max_area_ratio'])
medium_max = float(config['bbox_size']['medium_max_area_ratio'])
combined_rows = []
for row in detailed.prediction_details.to_dict(orient='records'):
    combined_rows.append({
        'source_type': 'prediction', 'image_id': row['image_id'],
        'object_id': row['prediction_id'], 'counterpart_id': row.get('matched_gt_id', ''),
        'class_id': int(row['class_id']), 'confidence': float(row['confidence']),
        'x1': float(row['x1']), 'y1': float(row['y1']), 'x2': float(row['x2']), 'y2': float(row['y2']),
        'bbox_area_ratio': float(row['bbox_area_ratio']),
        'bbox_size': core.categorize_bbox_size(float(row['bbox_area_ratio']), small_max=small_max, medium_max=medium_max),
        'threshold': float(row['threshold']), 'best_iou': float(row['best_iou']),
        'best_raw_confidence': float(row['confidence']), 'match_status': row['match_status'],
        'error_type': row['error_type'],
    })
for row in detailed.ground_truth_details.to_dict(orient='records'):
    raw_conf = row.get('best_raw_confidence')
    combined_rows.append({
        'source_type': 'ground_truth', 'image_id': row['image_id'],
        'object_id': row['gt_id'], 'counterpart_id': row.get('matched_prediction_id', ''),
        'class_id': int(row['class_id']), 'confidence': None,
        'x1': float(row['x1']), 'y1': float(row['y1']), 'x2': float(row['x2']), 'y2': float(row['y2']),
        'bbox_area_ratio': float(row['bbox_area_ratio']),
        'bbox_size': core.categorize_bbox_size(float(row['bbox_area_ratio']), small_max=small_max, medium_max=medium_max),
        'threshold': float(row['threshold']), 'best_iou': float(row['best_iou']),
        'best_raw_confidence': None if pd.isna(raw_conf) else float(raw_conf),
        'match_status': row['match_status'], 'error_type': row['error_type'],
    })
matches_df = pd.DataFrame(combined_rows).sort_values(
    ['image_id', 'source_type', 'object_id'], kind='mergesort'
).reset_index(drop=True)
errors_df = matches_df[matches_df['error_type'] != 'true_positive'].copy()

error_summary_df = (
    errors_df.groupby(['source_type', 'error_type'], dropna=False)
    .agg(affected_cases=('object_id', 'count'), affected_images=('image_id', 'nunique'))
    .reset_index().sort_values(['source_type', 'error_type'], kind='mergesort')
)
bbox_summary_df = (
    errors_df.groupby(['source_type', 'bbox_size', 'error_type'], dropna=False)
    .agg(affected_cases=('object_id', 'count'), affected_images=('image_id', 'nunique'))
    .reset_index().sort_values(['source_type', 'bbox_size', 'error_type'], kind='mergesort')
)

sweep_df.to_csv(METRICS_DIR / 'validation_threshold_sweep.csv', index=False, encoding='utf-8-sig')
operating_points_df.to_csv(METRICS_DIR / 'validation_operating_points.csv', index=False, encoding='utf-8-sig')
error_summary_df.to_csv(METRICS_DIR / 'validation_error_summary.csv', index=False, encoding='utf-8-sig')
bbox_summary_df.to_csv(METRICS_DIR / 'validation_error_by_bbox_size.csv', index=False, encoding='utf-8-sig')
matches_df.to_csv(RECORDS_DIR / 'validation_matches.csv', index=False, encoding='utf-8-sig')
errors_df.to_csv(RECORDS_DIR / 'validation_error_cases.csv', index=False, encoding='utf-8-sig')

import matplotlib.pyplot as plt
figure = plt.figure(figsize=(10, 6))
plt.plot(sweep_df['threshold'], sweep_df['precision'], marker='o', label='Precision')
plt.plot(sweep_df['threshold'], sweep_df['recall'], marker='o', label='Recall')
plt.plot(sweep_df['threshold'], sweep_df['f1'], marker='o', label='F1')
plt.xlabel('Confidence threshold')
plt.ylabel('Metric')
plt.title('FleetVision 04.5K Validation Threshold Sweep')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
figure.savefig(METRICS_DIR / 'validation_threshold_sweep.png', dpi=160)
plt.close(figure)

print(operating_points_df[['operating_point', 'threshold', 'precision', 'recall', 'f1']].to_string(index=False))
print('Balanced threshold：', balanced_threshold)
print('Detailed errors：', len(errors_df))
print('CELL 6 PASS')


In [ ]:
# Cell 7
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 6 後新增
# 主要目的：建立人工複核 worklist、代表性 overlay、資料改善排序與分析報告。
# 執行前提：Cell 6 PASS。
# 重點：只提出改善優先順序，不修改 annotation、不重新訓練。
# 驗收標準：產生 review worklist、最多 60 張 overlays、priorities CSV 與 Markdown report，顯示 CELL 7 PASS。

from PIL import Image, ImageDraw

priority_order = {
    'no_detection': 0,
    'low_confidence_miss': 1,
    'localization_miss': 2,
    'localization_error': 2,
    'background_false_positive': 3,
    'duplicate_prediction': 4,
}
review_rows = []
for image_id, group in errors_df.groupby('image_id', sort=True):
    types = sorted(set(group['error_type'].astype(str)), key=lambda value: (priority_order.get(value, 9), value))
    review_rows.append({
        'image_id': image_id,
        'primary_error_type': types[0] if types else '',
        'error_types': '|'.join(types),
        'error_case_count': int(len(group)),
        'ground_truth_error_count': int((group['source_type'] == 'ground_truth').sum()),
        'prediction_error_count': int((group['source_type'] == 'prediction').sum()),
        'review_status': 'pending',
        'human_error_category': '',
        'review_notes': '',
    })
review_worklist_df = pd.DataFrame(review_rows)
if not review_worklist_df.empty:
    review_worklist_df['_priority'] = review_worklist_df['primary_error_type'].map(priority_order).fillna(9)
    review_worklist_df = review_worklist_df.sort_values(
        ['_priority', 'error_case_count', 'image_id'], ascending=[True, False, True], kind='mergesort'
    ).drop(columns='_priority').reset_index(drop=True)

max_overlays = int(config.get('review', {}).get('max_overlay_images', 60))
selected_images = review_worklist_df['image_id'].head(max_overlays).tolist() if not review_worklist_df.empty else []
for image_id in selected_images:
    source_path = next(path for path in image_paths if path.name == image_id)
    image = Image.open(source_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    for row in ground_truth_df[ground_truth_df['image_id'] == image_id].itertuples(index=False):
        draw.rectangle((row.x1, row.y1, row.x2, row.y2), outline='lime', width=3)
        draw.text((row.x1, max(0, row.y1 - 12)), 'GT', fill='lime')
    active = predictions_df[
        (predictions_df['image_id'] == image_id) & (predictions_df['confidence'] >= balanced_threshold)
    ]
    for row in active.itertuples(index=False):
        draw.rectangle((row.x1, row.y1, row.x2, row.y2), outline='red', width=3)
        draw.text((row.x1, max(0, row.y1 - 12)), f'P {row.confidence:.2f}', fill='red')
    overlay_path = OVERLAYS_DIR / f'{Path(image_id).stem}_overlay.jpg'
    image.save(overlay_path, quality=92)

if not review_worklist_df.empty:
    review_worklist_df['overlay_relative_path'] = review_worklist_df['image_id'].map(
        lambda value: f'overlays/{Path(value).stem}_overlay.jpg' if value in selected_images else ''
    )
else:
    review_worklist_df['overlay_relative_path'] = pd.Series(dtype=str)
review_worklist_df.to_csv(REVIEW_DIR / 'validation_error_review_worklist.csv', index=False, encoding='utf-8-sig')

priority_input = errors_df[['error_type', 'image_id', 'object_id']].copy()
priorities_df = core.build_data_improvement_priorities(priority_input)
priorities_df.to_csv(REPORTS_DIR / 'data_improvement_priorities.csv', index=False, encoding='utf-8-sig')

balanced_row = operating_points_df[operating_points_df['operating_point'] == 'balanced'].iloc[0]
report_lines = [
    '# FleetVision Phase 04.5K Baseline Error Analysis', '',
    '## Gate boundary', '',
    '- Analysis split: `valid` only',
    '- Test set used for tuning: `false`',
    '- Test set read by this Notebook: `false`',
    '- Training started: `false`',
    '- Deployment acceptance: `NOT_YET_APPROVED`', '',
    '## Baseline context', '',
    f'- Model SHA256: `{EXPECTED_BEST_PT_SHA256}`',
    '- Model family: `YOLOv8s Detect`',
    '- Class: `damage`',
    f'- Validation images / GT: `{EXPECTED_VALID_IMAGES}` / `{EXPECTED_VALID_ANNOTATIONS}`', '',
    '## Balanced validation candidate', '',
    f"- Threshold: `{float(balanced_row['threshold']):.2f}`",
    f"- Precision: `{float(balanced_row['precision']):.6f}`",
    f"- Recall: `{float(balanced_row['recall']):.6f}`",
    f"- F1: `{float(balanced_row['f1']):.6f}`", '',
    '## Candidate operating points', '',
]
for row in operating_points_df.to_dict(orient='records'):
    report_lines.append(
        f"- `{row['operating_point']}`: threshold={float(row['threshold']):.2f}, "
        f"P={float(row['precision']):.6f}, R={float(row['recall']):.6f}, F1={float(row['f1']):.6f}"
    )
report_lines.extend(['', '## Error taxonomy', ''])
for row in error_summary_df.to_dict(orient='records'):
    report_lines.append(
        f"- `{row['source_type']} / {row['error_type']}`: cases={int(row['affected_cases'])}, "
        f"images={int(row['affected_images'])}"
    )
report_lines.extend([
    '', '## Decision boundary', '',
    'The three values are validation threshold candidates only. They do not approve deployment.',
    'The next step is human review of representative errors and data-improvement prioritization.',
    'No annotation was modified and no retraining was performed.',
])
(REPORTS_DIR / 'baseline_error_analysis.md').write_text(
    '\n'.join(report_lines) + '\n', encoding='utf-8', newline='\n'
)

print('Review cases：', len(review_worklist_df))
print('Overlay images：', len(selected_images))
print('Priority categories：', len(priorities_df))
print('CELL 7 PASS')


In [ ]:
# Cell 8
# 操作類型：新增並執行
# 插入或覆蓋位置：Cell 7 後新增
# 主要目的：完成安全稽核、artifact manifest、Gate result 與小型 ZIP Log。
# 執行前提：Cell 7 PASS。
# 重點：確認只解壓 validation；ZIP 不含 best.pt；classification 固定為核准的 04.5K 完成狀態。
# 驗收標準：顯示 CELL 8 PASS、ZIP 路徑與 SHA256；test_set_used_for_tuning=False、training_started=False。

for path in EXTRACT_ROOT.rglob('*'):
    if not path.is_file():
        continue
    relative = '/' + path.relative_to(EXTRACT_ROOT).as_posix().lower() + '/'
    if '/train/' in relative or '/test/' in relative:
        raise RuntimeError(f'04.5K 安全違規：解壓到非 validation 路徑 {relative}')

ENVIRONMENT_PATH = SESSION_ROOT / 'environment.json'
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'ultralytics': importlib.metadata.version('ultralytics'),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0),
}
ENVIRONMENT_PATH.write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8')

GATE_RESULT_PATH = SESSION_ROOT / '04_5K_gate_result.json'
gate_result = {
    'gate_id': '04.5K',
    'outcome': 'PASS',
    'classification': 'VALIDATION_ERROR_ANALYSIS_AND_THRESHOLD_CANDIDATES_COMPLETED',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_04_5j_gate_result': str(J_GATE_RESULT_PATH),
    'source_04_5j_classification': EXPECTED_J_CLASSIFICATION,
    'best_pt_sha256': EXPECTED_BEST_PT_SHA256,
    'allowed_split': 'valid',
    'validation_image_count': EXPECTED_VALID_IMAGES,
    'validation_ground_truth_instances': EXPECTED_VALID_ANNOTATIONS,
    'raw_prediction_count': int(len(predictions_df)),
    'threshold_candidate_count': int(len(operating_points_df)),
    'balanced_threshold': balanced_threshold,
    'test_set_used_for_tuning': False,
    'test_set_read': False,
    'training_started': False,
    'annotation_modified': False,
    'deployment_acceptance': 'NOT_YET_APPROVED',
}
GATE_RESULT_PATH.write_text(json.dumps(gate_result, ensure_ascii=False, indent=2), encoding='utf-8')

ARTIFACT_MANIFEST_PATH = MANIFEST_DIR / 'phase_04_5k_artifact_manifest.csv'
artifact_rows = []
for path in sorted(p for p in SESSION_ROOT.rglob('*') if p.is_file()):
    if path.suffix.lower() == '.pt':
        raise RuntimeError(f'04.5K output 不得包含權重：{path}')
    artifact_rows.append({
        'relative_path': path.relative_to(SESSION_ROOT).as_posix(),
        'size_bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })
with ARTIFACT_MANIFEST_PATH.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['relative_path', 'size_bytes', 'sha256'])
    writer.writeheader()
    writer.writerows(artifact_rows)

CHECKSUM_PATH = MANIFEST_DIR / 'phase_04_5k_checksums.sha256'
checksum_lines = []
for path in sorted(p for p in SESSION_ROOT.rglob('*') if p.is_file() and p != CHECKSUM_PATH):
    checksum_lines.append(f'{sha256_file(path)}  {path.relative_to(SESSION_ROOT).as_posix()}')
CHECKSUM_PATH.write_text('\n'.join(checksum_lines) + '\n', encoding='ascii')

zip_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
zip_id = hashlib.sha256(
    f'{BUNDLE_ID}|{zip_stamp}|{EXPECTED_BEST_PT_SHA256}|{balanced_threshold}'.encode('utf-8')
).hexdigest()[:8]
RESULT_ZIP = SESSION_ROOT / f'04_5K_{zip_stamp}_{zip_id}_ZIP_LOG.zip'
zip_candidates = sorted(p for p in SESSION_ROOT.rglob('*') if p.is_file() and p != RESULT_ZIP)
with zipfile.ZipFile(RESULT_ZIP, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in zip_candidates:
        if path.suffix.lower() == '.pt':
            raise RuntimeError(f'ZIP 禁止包含權重：{path}')
        archive.write(path, arcname=path.relative_to(SESSION_ROOT).as_posix())
with zipfile.ZipFile(RESULT_ZIP, mode='r') as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'ZIP CRC failure：{bad_member}')
    names = archive.namelist()
    if any(name.lower().endswith('.pt') for name in names):
        raise RuntimeError('ZIP 內出現禁止的 .pt 權重')

print(json.dumps(gate_result, ensure_ascii=False, indent=2))
print('ZIP：', RESULT_ZIP)
print('ZIP SHA256：', sha256_file(RESULT_ZIP))
print('CELL 8 PASS')
